# Sesión 1 · Herramientas y el bucle

**Prácticas de *LLMs aplicados a Finanzas* · MIAX · jueves 10 de septiembre**

## Antes de empezar: por qué no vamos a montar un RAG

Lo esperable, en un curso de LLMs sobre documentos, sería montar un RAG
clásico: trocear los informes, calcular sus *embeddings*, guardarlos en un
índice vectorial y, ante cada pregunta, recuperar los *k* fragmentos más
parecidos y metérselos al modelo en el prompt. Recuperar, y luego generar. Es
el patrón que describe el paper de 2020 que le puso nombre
([Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP
Tasks*](https://arxiv.org/abs/2005.11401)) y sigue siendo el punto de partida
razonable para casi cualquier sistema de este tipo.

Vamos a construir todas esas piezas. Lo que no vamos a hacer es dejar que sean
**la arquitectura**, y conviene decir por qué.

Un RAG clásico decide **de antemano** qué información necesita el modelo. La
recuperación ocurre siempre, una sola vez, antes de generar, con la pregunta
tal y como llegó y con una *k* fijada de antemano. Eso funciona mientras la
pregunta se parezca a un párrafo del corpus. Deja de funcionar en cuanto:

- **la respuesta no está en un sitio, sino en dos.** «¿Qué riesgos añadió
  Microsoft entre FY2024 y FY2025?» exige recuperar dos veces y comparar. Una
  sola pasada de recuperación devuelve fragmentos de los dos años mezclados y
  el modelo se los inventa a medias;
- **el dato exacto no está en la prosa, sino en una tabla estructurada.**
  Buscar «beneficio neto de Apple» por similitud semántica devuelve párrafos
  que *hablan* del beneficio neto. La cifra auditada está en el XBRL, y no
  hace falta buscarla: se consulta;
- **la pregunta se refiere a algo que no existe.** Un RAG plano siempre
  devuelve sus *k* fragmentos, aunque la compañía por la que preguntáis no
  esté en el corpus. Siempre recupera algo, y ese algo siempre parece una
  respuesta.

La alternativa no es tirar el retrieval: es **bajarlo de arquitectura a
herramienta**. En lugar de un *pipeline* fijo que recupera y luego genera,
un modelo con varias herramientas que decide sobre la marcha cuál usar,
cuántas veces y con qué consulta. La documentación de LangChain llama a lo
primero *2-Step RAG* y a lo segundo *Agentic RAG*, y compara los dos en
[docs.langchain.com/oss/python/langchain/retrieval](https://docs.langchain.com/oss/python/langchain/retrieval).

Y el retrieval sobrevive ahí dentro por dos razones que no tienen nada que ver
con lo que el modelo sea capaz de leer:

- **Coste.** El corpus son 649.119 tokens. Meterlo entero en cada pregunta es
  pagarlo entero en cada pregunta. Lo calculáis vosotros en §2.
- **Auditabilidad.** En un entorno regulado hay que poder señalar el párrafo
  del que sale la cifra. El contexto largo da la respuesta; el retrieval da el
  ancla.

*(RAG lo visteis en teoría el sábado 12. Esto es lo que se hace con ello.)*

## Qué construimos hoy

Un **agente investigador sobre informes 10-K de la SEC**. Al final de la
sesión tendrá cuatro herramientas y sabrá elegir entre ellas:

| Herramienta | Para qué |
| --- | --- |
| `list_available()` | Comprobar qué hay en el corpus antes de inventárselo |
| `get_xbrl_fact()` | La cifra exacta, tal y como la reportó la compañía |
| `search_filings()` | Buscar en el texto de los informes. Hoy es caja negra |
| `read_section()` | El texto completo de una sección. Cara |

Esas cuatro herramientas son las tres carencias de arriba, resueltas. El
agente puede **recuperar dos veces** y comparar, porque quien decide cuántas
búsquedas hacen falta es él y no el *pipeline*. Puede **no buscar**, y
consultar la tabla XBRL cuando lo que se le pide es una cifra exacta. Y puede
**comprobar que algo existe** antes de responder, en lugar de devolver los
cinco fragmentos más parecidos a una pregunta sobre una empresa que no está.

El precio de esa flexibilidad es que el sistema se vuelve menos predecible: un
*pipeline* fijo siempre hace lo mismo, y un agente no. Por eso la sesión que
viene va entera de medirlo —evaluación de trayectoria, no solo de respuesta— y
de ponerle límites. Hoy toca que funcione; el día 17, que aguante.

Fijaos en la asimetría que hay en la tabla, porque es el eje de todo el curso:
`get_xbrl_fact` es barata, exacta y determinista, y `search_filings` es cara,
difusa y aproximada. **Elegir bien entre las dos es el trabajo del agente**, y
es lo que se evalúa. Un agente que acierta la cifra leyéndola de la prosa está
mal aunque el número salga bien: la próxima vez, con otra tabla partida, saldrá
mal y nadie se enterará.

## Qué se entrega el 24

Repositorio, golden set de 20 preguntas vuestras con al menos 6 comparativas,
informe con la tabla *baseline* contra final, y una presentación de 8 minutos
en la que ejecutáis **10 preguntas ciegas** que no veis hasta ese día. Todo
está en el enunciado que tenéis en la mano.

## Un aviso sobre el orden del curso

Hoy vais a escribir un bucle ReAct y a recuperar texto de un corpus. **El
viernes 18 os explicarán ReAct y el paper; RAG lo visteis el sábado 12.** Es
deliberado: hoy construís, y la teoría llega después a ponerle nombre a algo
que ya habréis tocado con las manos.

## Cómo se usa este notebook

- **Se ejecuta de arriba abajo, sin saltos.** No hay estado oculto: si saltáis
  una celda, la siguiente falla.
- Cada sección termina con `assert`. Si pasan, podéis seguir.
- **Ninguna clave está escrita aquí dentro.** Se piden por `getpass`.
- Lo que puede fallar por red está en `try/except` y degrada a un modo sin esa
  función. Nada de esto debería parar la clase.

Junto al proyecto necesitáis `modulos/miax_s1.py`,
`modulos/demo_traza.json` y los dos ZIP del corpus, que se montan en §1.

In [1]:
# El agente terminado, antes de construirlo. Cinco minutos y ninguna
# explicación: esto es el destino, no el camino.
#
# Reproduce una ejecución grabada. Todavía no hay nada instalado ni montado.
#
# NOTA DEL GRUPO: el `demo_traza.json` que se repartió es PROVISIONAL —lo
# dice su propio campo `metadatos.origen`— y le falta la clave
# `metadatos.fecha` que `miax_s1.demo_apertura()` lee al imprimir el pie, así
# que sin regenerarlo esta celda termina en un KeyError. El aviso que imprime
# `miax_s1.py` dice qué hacer: `python modulos/generar_traza_demo.py`. Ese
# script no venía con el material; está en `modulos/` y sustituye la
# grabación por una ejecución real de nuestro agente.
import sys
from pathlib import Path

RAIZ = Path.cwd()
for carpeta in (RAIZ, RAIZ / "modulos"):
    if str(carpeta) not in sys.path:
        sys.path.insert(0, str(carpeta))

try:
    import miax_s1
    _demo = miax_s1.demo_apertura()
except Exception as e:
    print(f"No se pudo reproducir la demo ({type(e).__name__}: {e}).")
    print("Regenera la traza con:  python modulos/generar_traza_demo.py")
    print("(necesita la clave en api_key.txt y el corpus del notebook 00)")

PREGUNTA: ¿Qué riesgos nuevos relacionados con la inteligencia artificial añadió Microsoft en su 10-K de FY2025 respecto al de FY2024, y cuánto creció su revenue entre esos dos ejercicios?

TRAYECTORIA
  1. list_available()
       -> COMPAÑÍAS Y EJERCICIOS DISPONIBLES
       -> AAPL — Apple Inc.
  2. get_xbrl_fact(ticker='MSFT', fiscal_year=2025, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 281,724,000,000 USD (cierre de e
  3. get_xbrl_fact(ticker='MSFT', fiscal_year=2024, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 245,122,000,000 USD (cierre de e
  4. search_filings(query='artificial intelligence', ticker='MSFT', fiscal_year=2025, item='1A', k=10)
       -> [MSFT-2025-1A-0004] MSFT FY2025 Item 1A (similitud 0.243)
       -> and sales incentives, which could adversely affect our financial condi

> ## Nota del grupo: qué se ha cambiado respecto al material de clase, y por qué
>
> Este notebook es el de la sesión 1 con los ejercicios resueltos. Los cambios
> respecto al original son cuatro, y los cuatro están marcados en el código con
> un comentario `CAMBIO RESPECTO AL MATERIAL DE CLASE`:
>
> | Cambio | Motivo |
> | --- | --- |
> | OpenRouter → **OpenAI** | Es la cuenta de la que disponemos. `init_chat_model` abstrae el proveedor, así que solo cambia una cadena de texto: el resto del notebook no se entera, que es justamente lo que la celda del proveedor enseña. |
> | `getpass` → **`api_key.txt`** | `getpass` necesita a alguien delante. La entrega tiene que poder ejecutarse de un tirón, sin nadie tecleando, tanto para regenerar los resultados como para las diez preguntas ciegas del día 24. El fichero está en `.gitignore`. |
> | Descompresión de ZIP → **`construir_corpus()`** | En este proyecto solo llegó `indice_faiss.zip`. `construir_corpus()` usa los ZIP si están y, si no, reconstruye el corpus y lo verifica contra los 13 anclas y las 12 cifras del golden set oficial. Todo el detalle está en `00_Preparacion_del_corpus.ipynb`. |
> | Modelo fijo → **autodetección** | No controlamos a qué modelos da acceso la cuenta. Se prueban `gpt-5-mini`, `gpt-4.1-mini` y `gpt-4o-mini` en orden y se usa el primero disponible, con el elegido registrado junto a los resultados. |
>
> Todo lo demás —el orden de las celdas, las explicaciones, las
> verificaciones `§n` y las versiones fijadas de las dependencias— se conserva
> tal cual. La lógica definitiva del entregable vive en el paquete `agente/`;
> este notebook es el recorrido didáctico que lleva hasta ella.

In [2]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
#
# CAMBIO RESPECTO AL MATERIAL DE CLASE: se instala `langchain-openai` en lugar
# de `langchain-openrouter`. El motivo está en la celda de justificación de
# debajo. Las versiones de todo lo demás son exactamente las de clase.
#
# Se instala desde `requirements.txt` y no con una lista escrita aquí, para
# que el notebook y el repositorio no puedan desincronizarse.
%pip install -q -r requirements.txt
print("Instalación terminada.")

Note: you may need to restart the kernel to use updated packages.
Instalación terminada.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Por qué van fijadas las versiones

Porque LangChain publica cada pocos días y una API que se mueve debajo de un
notebook lo rompe sin tocar una línea de código.

No es una precaución teórica: **los notebooks de la edición anterior de este
curso ya no ejecutan.** `create_react_agent`, `MemorySaver`,
`with_structured_output` y `RunnableWithMessageHistory` eran la API correcta
hace un año y hoy son otra cosa. Lo que veis aquí está verificado contra la
documentación oficial el 2 de septiembre de 2026.

De ahí salen dos hábitos que valen para cualquier proyecto que dependa de un
proveedor de modelos:

1. **Fijad la versión exacta**, no el rango. `>=1.3` no es una versión.
2. **Anotad la fecha de verificación** al lado del pin, para saber cómo de
   viejo es lo que estáis leyendo.

In [3]:
# Claves. Nunca escritas en el notebook.
#
# CAMBIO RESPECTO AL MATERIAL DE CLASE: en clase se piden por teclado con
# `getpass`. Aquí se leen de `api_key.txt`, un fichero de la raíz del proyecto
# que está en `.gitignore`. El motivo: `getpass` necesita a alguien delante, y
# la entrega tiene que poder ejecutarse de un tirón —`jupyter nbconvert
# --execute`, o la evaluación de las diez preguntas ciegas del día 24— sin que
# nadie teclee nada.
#
# El orden de búsqueda es: variable de entorno OPENAI_API_KEY, y si no está,
# `api_key.txt`. La clave no aparece escrita en ningún notebook ni en ningún
# módulo, que es lo que exige el §5 del enunciado.
import sys
from pathlib import Path

RAIZ = Path.cwd()
for carpeta in (RAIZ, RAIZ / "modulos"):
    if str(carpeta) not in sys.path:
        sys.path.insert(0, str(carpeta))

from agente import config

# Y se comprueba algo más que la existencia de la clave: que el modelo
# RESPONDA. No son lo mismo, y confundirlos cuesta caro. Una clave válida
# sobre una cuenta sin crédito devuelve un 429 y una organización sin acceso a
# la familia 5 devuelve un 404; en los dos casos «hay clave» y la primera
# invocación real revienta, normalmente a mitad de una evaluación y después de
# haber pagado las primeras preguntas. `hay_modelo()` gasta una llamada de
# cuatro tokens y lo averigua antes.
HAY_CLAVE = config.hay_modelo()

if HAY_CLAVE:
    print(f"Modelo de OpenAI disponible: {config.modelo_por_defecto()}")
else:
    print(config.motivo_sin_modelo())
    print("\nLas celdas que llaman al modelo se van a saltar solas. Las de "
          "datos, herramientas y retrieval funcionan igual: el notebook se "
          "puede recorrer entero sin gastar un céntimo.")

# Una sola clave. La observabilidad de este curso no necesita ninguna más:
# la trayectoria se imprime en el propio notebook (§6).

Modelo de OpenAI disponible: gpt-5-mini


In [4]:
# Corpus e índice.
#
# CAMBIO RESPECTO AL MATERIAL DE CLASE: la celda original descomprime dos ZIP
# y verifica su SHA-256 contra el manifiesto. Aquí se llama a
# `construir_corpus()`, que hace exactamente eso **si los ZIP están**, y si no
# reconstruye el corpus desde el índice y desde la API pública de la SEC.
#
# El motivo no es preferencia: en este proyecto solo llegó `indice_faiss.zip`.
# El notebook `00_Preparacion_del_corpus.ipynb` explica la reconstrucción
# entera y la verifica contra los 13 anclas del golden set oficial y contra
# las 12 cifras que declara. Aquí basta con saber que deja los mismos ficheros
# en el mismo sitio.
from agente import corpus

corpus.construir_corpus()

print()
for p in sorted(config.DIR_CORPUS.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(config.DIR_CORPUS)):28s} "
              f"{p.stat().st_size / 1e6:7.2f} MB")

El corpus ya está montado en C:\Users\jdmar\Desktop\Taller NLP\corpus. Nada que hacer.

  chunks.jsonl                    3.81 MB
  indice\chunks_meta.parquet      1.48 MB
  indice\corpus.faiss             2.69 MB
  indice\MANIFEST.md              0.00 MB
  MANIFIESTO.md                   0.00 MB
  secciones.jsonl                 3.20 MB
  xbrl_facts.parquet              0.01 MB


In [5]:
# Un proveedor es una cadena de texto.
#
# `init_chat_model` devuelve el mismo objeto sea cual sea el proveedor: el
# resto del notebook no se entera de cuál hay debajo. Cambiad la cadena y
# todo lo demás sigue igual. Es justo lo que hacemos aquí: donde clase pone
# `openrouter:google/gemini-3.8-flash`, nosotros ponemos `openai:...`, y no
# cambia una línea más.
from langchain.chat_models import init_chat_model

# El modelo no se fija a ciegas: se prueban en orden gpt-5-mini, gpt-4.1-mini
# y gpt-4o-mini, y se usa el primero al que la cuenta tenga acceso. El
# resultado se cachea en `.cache/`. El motivo es operativo: no controlamos a
# qué modelos da acceso la cuenta que ejecute esto, y un NotFoundError a mitad
# de la evaluación del día 24 no es un riesgo aceptable.
if HAY_CLAVE:
    NOMBRE_MODELO = config.modelo_por_defecto()
    MODELO = f"openai:{NOMBRE_MODELO}"
    modelo = config.crear_modelo()
    print(f"Modelo detectado: {MODELO}")
    print(modelo.invoke("Responde solo con la palabra: listo").text)
else:
    NOMBRE_MODELO, MODELO, modelo = None, None, None
    print("Sin clave: no hay modelo. El resto del notebook se puede leer.")

# El mismo código contra otros proveedores. No los ejecutamos: cuestan dinero
# y hacen falta otras tantas claves. El punto es que solo cambia la cadena.
#
#   init_chat_model("openai:gpt-5-mini",                  temperature=0)
#   init_chat_model("anthropic:claude-opus-5",            temperature=0)
#   init_chat_model("google_genai:gemini-3.8-flash",      temperature=0)
#   init_chat_model("openrouter:auto",                    temperature=0)
#
# El último deja que OpenRouter elija modelo. Sirve para enseñar el concepto y
# NO sirve para evaluar: si el modelo cambia entre dos ejecuciones, la
# comparación baseline-contra-final no significa nada. Por la misma razón
# nosotros cacheamos el modelo detectado en lugar de redetectarlo cada vez.
#
# `temperature=0` va en todo lo que se vaya a evaluar. Con temperatura alta,
# dos ejecuciones de la misma pregunta dan métricas distintas y no sabéis si
# mejorasteis el sistema o tuvisteis suerte. Matiz que nos ha costado un 400:
# la familia gpt-5 SOLO admite el valor por defecto, así que
# `config.crear_modelo()` fija `temperature=0` únicamente donde se admite y lo
# deja registrado en los resultados.

# --- verificación de §1 --------------------------------------------------
assert config.RUTA_CHUNKS.is_file(), \
    "El corpus no está montado: repasad la celda de setup."
assert config.RUTA_FAISS.is_file(), \
    "Falta el índice FAISS: ejecutad 00_Preparacion_del_corpus.ipynb."
print("§1 listo.")

Modelo detectado: openai:gpt-5-mini


listo
§1 listo.


## Anatomía de un 10-K

El 10-K es el informe anual que toda empresa cotizada en EE. UU. presenta ante
la SEC. Es un documento normalizado: los mismos epígrafes, en el mismo orden,
todos los años y en todas las compañías. Eso es lo que lo hace utilizable como
corpus.

Nos quedamos con cuatro epígrafes, que son donde está lo que se puede
preguntar:

| Item | Qué contiene | Qué se le pregunta |
| --- | --- | --- |
| **1A** · Risk Factors | Los riesgos que la compañía declara | Qué riesgos nuevos aparecen, cómo cambian entre ejercicios |
| **7** · MD&A | La dirección explicando sus propios resultados | Por qué subió o bajó una magnitud |
| **7A** · Market Risk | Exposición a tipos, divisa y precios | Cuantitativo y corto |
| **8** · Financial Statements | Los estados financieros y sus notas | Cifras, y de dónde salen |

Dos cosas que hay que saber del corpus antes de tocarlo:

**`fiscal_year` no es el año de presentación.** Las seis compañías cierran
ejercicio en cuatro meses distintos —NVDA en enero, MSFT en junio, AAPL en
septiembre, y GOOGL, META y AMZN en diciembre—, y está elegido así a
propósito. El 10-K de NVDA FY2025 se presentó en febrero de 2025; el de
Alphabet FY2025, en febrero de **2026**. Quien razone por fecha de
presentación se equivoca.

**NVIDIA no pone sus estados financieros bajo el Item 8.** Los deja bajo el
Item 15 y en el 8 escribe una remisión de dos líneas. El corpus sirve el
contenido correcto bajo la clave `"8"` y deja constancia en el campo
`item_origen`. Si no lo hiciera, `read_section("NVDA", 2025, "8")` devolvería
cuarenta tokens inútiles.

In [6]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json

import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open(config.RUTA_SECCIONES, encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")

item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Informe mayor: 80,764 · menor: 32,088
Sección mayor: META FY2025 Item 1A con 34,751 tokens


In [7]:
# Los tres caminos hacia el mismo dato, con su precio.
#
# CAMBIO RESPECTO AL MATERIAL DE CLASE: los precios son los de OpenAI y salen
# de `agente/config.py`, no de una tabla escrita aquí. Que estén en un solo
# sitio es lo que permite que el coste de este notebook, el del notebook 05 y
# el del informe se calculen con los mismos números; si hubiera tres tablas,
# tres meses después serían tres tablas distintas.
PRECIOS = config.PRECIOS_OPENAI


def coste(entrada: int, salida: int, modelo: str) -> float:
    """Coste en dólares de una llamada, dados los tokens de cada lado."""
    p_in, p_out = config.precio_de(modelo)
    return entrada / 1e6 * p_in + salida / 1e6 * p_out


MODELO_COSTE = NOMBRE_MODELO or "gpt-5-mini"
SALIDA_TIPICA = 300      # tokens de respuesta, aproximadamente

# Tomamos una pregunta real: algo sobre el 10-K de Microsoft de FY2025.
informe = int(
    secciones[(secciones.ticker == "MSFT")
              & (secciones.fiscal_year == 2025)].n_tokens.sum()
)
fragmentos = 5 * 402     # cinco fragmentos, de unos 402 tokens de media
consulta_xbrl = 40       # ticker, ejercicio, concepto y el valor devuelto

caminos = [
    ("Informe entero en contexto", informe),
    ("Cinco fragmentos recuperados", fragmentos),
    ("Una consulta a XBRL", consulta_xbrl),
]

print(f"Modelo: {MODELO_COSTE}  "
      f"({config.precio_de(MODELO_COSTE)[0]} $/M entrada, "
      f"{config.precio_de(MODELO_COSTE)[1]} $/M salida)")
print(f"Precios consultados el {config.FECHA_PRECIOS}. REVISAR LA VÍSPERA.\n")
print("Camino                             entrada   $ entrada    $ total")
for nombre, tokens in caminos:
    solo_entrada = coste(tokens, 0, MODELO_COSTE)
    total = coste(tokens, SALIDA_TIPICA, MODELO_COSTE)
    print(f"{nombre:32s} {tokens:9,} {solo_entrada:11.5f} {total:10.5f}")

# Dos lecturas distintas de la misma tabla, y las dos hacen falta:
caro, barato = caminos[0][1], caminos[2][1]
print(f"\nEn tokens de ENTRADA, el camino caro cuesta {caro / barato:,.0f} "
      f"veces más que la consulta a XBRL.")
razon = (coste(caro, SALIDA_TIPICA, MODELO_COSTE)
         / coste(barato, SALIDA_TIPICA, MODELO_COSTE))
print(f"Por pregunta completa, con {SALIDA_TIPICA} tokens de respuesta, "
      f"cuesta {razon:.0f} veces más: la respuesta se paga igual por los "
      f"tres caminos.")

# Y esa diferencia se multiplica por el número de preguntas y de grupos.
PREGUNTAS, GRUPOS = 40, 10
for nombre, tokens in caminos:
    c = coste(tokens, SALIDA_TIPICA, MODELO_COSTE) * PREGUNTAS * GRUPOS
    print(f"  {PREGUNTAS} preguntas x {GRUPOS} grupos, {nombre.lower()}: "
          f"{c:.2f} $")

# Y ahora la pregunta que de verdad importa, la comparativa entre dos
# ejercicios: hay que traerse los dos informes.
comparativa = int(
    secciones[(secciones.ticker == "MSFT")
              & (secciones.fiscal_year.isin([2024, 2025]))].n_tokens.sum()
)
todo = int(secciones.n_tokens.sum())
print(f"\nComparar FY2024 con FY2025 metiendo los dos informes enteros: "
      f"{comparativa:,} tokens = "
      f"{coste(comparativa, SALIDA_TIPICA, MODELO_COSTE):.4f} $")
print(f"Meter el corpus entero en cada pregunta: {todo:,} tokens = "
      f"{coste(todo, SALIDA_TIPICA, MODELO_COSTE):.4f} $")

# --- verificación de §2 --------------------------------------------------
assert len(secciones) == 48, "El corpus debería tener 48 secciones."
assert informe > fragmentos > consulta_xbrl
print("\n§2 listo.")

Modelo: gpt-5-mini  (0.25 $/M entrada, 2.0 $/M salida)
Precios consultados el 2026-09-21. REVISAR LA VÍSPERA.

Camino                             entrada   $ entrada    $ total
Informe entero en contexto          48,212     0.01205    0.01265
Cinco fragmentos recuperados         2,010     0.00050    0.00110
Una consulta a XBRL                     40     0.00001    0.00061

En tokens de ENTRADA, el camino caro cuesta 1,205 veces más que la consulta a XBRL.
Por pregunta completa, con 300 tokens de respuesta, cuesta 21 veces más: la respuesta se paga igual por los tres caminos.
  40 preguntas x 10 grupos, informe entero en contexto: 5.06 $
  40 preguntas x 10 grupos, cinco fragmentos recuperados: 0.44 $
  40 preguntas x 10 grupos, una consulta a xbrl: 0.24 $

Comparar FY2024 con FY2025 metiendo los dos informes enteros: 100,021 tokens = 0.0256 $
Meter el corpus entero en cada pregunta: 649,119 tokens = 0.1629 $

§2 listo.


## La pregunta

Los tres caminos llevan al mismo dato, y la tabla dice dos cosas distintas
según por dónde se mire.

En **tokens de entrada**, meter el informe entero cuesta más de mil veces lo
que cuesta preguntarle a XBRL. Pero por **pregunta completa** la diferencia se
queda en unas treinta veces, porque los trescientos tokens de respuesta se
pagan igual por los tres caminos. Merece la pena fijarse en eso: a escala
pequeña, los costes fijos disimulan la diferencia. A escala de la práctica
—cuarenta preguntas por diez grupos, varias veces mientras iteráis— deja de
disimularla.

Y aun así el modelo **puede** leerse el informe entero: le cabe en la ventana
de contexto. Así que la pregunta no es si cabe.

> **¿Por qué seguir recuperando fragmentos, si el modelo puede leerlo todo?**

Pensadla un minuto antes de seguir. Hay al menos dos respuestas buenas y una
de ellas no tiene nada que ver con el dinero.

*(Y hay una tercera pregunta debajo: si tenemos el dato exacto en una tabla
XBRL, ¿por qué íbamos a buscarlo en la prosa? Eso es el §3.)*

## Qué es una herramienta, y qué ve el modelo de ella

Una *tool* es una función de Python que el modelo puede pedir que se ejecute.
El decorador `@tool` la convierte en un esquema —nombre, parámetros con sus
tipos, y descripción— y ese esquema viaja en la petición junto a los mensajes.

Lo que hay que entender es **qué parte de vuestra función ve el modelo**:

| Lo ve | No lo ve |
| --- | --- |
| El nombre de la función | El cuerpo |
| Los nombres y tipos de los parámetros | Los comentarios |
| El *docstring*, entero | Cómo de rápida o cara es |
| Lo que devuelve, cuando la llama | Lo que hace por dentro |

De ahí sale la consecuencia que gobierna el resto de la sesión: **el modelo
decide si os llama leyendo el docstring**. Un docstring vago produce un agente
que elige mal, con el mismo código debajo. Volveremos a esto en el segundo
ejercicio.

Empezamos por la herramienta fácil: exacta, determinista y prácticamente
gratis.

In [8]:
from langchain.tools import tool

xbrl = pd.read_parquet(config.RUTA_XBRL)
print(f"{len(xbrl)} hechos XBRL · {xbrl.concept.nunique()} conceptos "
      f"distintos · {xbrl.ticker.nunique()} compañías")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE en lugar de
    leer un número del texto del informe.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto en taxonomía US-GAAP, p. ej. 'Revenues',
            'NetIncomeLoss', 'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    """
    filas = xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))
                 & (xbrl.concept == concept)]
    if filas.empty:
        disponibles = sorted(
            xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))].concept.unique()
        )
        if not disponibles:
            return (f"No hay datos de {ticker} para FY{fiscal_year} en el "
                    f"corpus. Usa list_available para ver qué hay.")
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(disponibles)}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
            f"(cierre de ejercicio {f.period_end}, según el {f.form})")


# Dos llamadas directas, sin modelo de por medio, para ver qué devuelve.
print(get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"}))
print(get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}))

# La segunda no es un fallo del corpus: Amazon no etiqueta GrossProfit en
# us-gaap. Que la herramienta lo diga en vez de devolver vacío es la
# diferencia entre un agente que contesta "no está" y uno que se lo inventa.

137 hechos XBRL · 14 conceptos distintos · 6 compañías
NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)
AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: Assets, EarningsPerShareDiluted, IncomeTaxExpenseBenefit, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, RevenueFromContractWithCustomerExcludingAssessedTax, StockholdersEquity


In [9]:
# La búsqueda en el texto de los informes.
#
# El cuerpo está en miax_s1.py y hoy es CAJA NEGRA a propósito: dentro hay
# troceado, embeddings, un índice FAISS y una decisión de top-k, y cada una de
# esas cuatro cosas se puede hacer mejor o peor. El día 17 se abre la caja, se
# mide lo que hace y se arregla tres veces.
#
# Hoy interesa otra cosa: que es la herramienta CARA y DIFUSA, la contraria de
# get_xbrl_fact. Devuelve texto que se parece a lo que pedisteis, no la
# respuesta.
import miax_s1


@tool
def search_filings(query: str, ticker: str | None = None,
                   fiscal_year: int | None = None,
                   item: str | None = None, k: int = 5) -> str:
    """Busca fragmentos de texto relevantes en los informes 10-K del corpus.

    Úsala para preguntas cualitativas: riesgos, estrategia, litigios,
    comentarios de la dirección. NO la uses para obtener cifras: para eso
    está get_xbrl_fact.

    Args:
        query: Qué buscar, en lenguaje natural.
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: Filtra por sección: '1A' riesgos, '7' MD&A,
            '7A' riesgo de mercado, '8' estados financieros.
        k: Número de fragmentos a devolver.

    Devuelve k fragmentos, cada uno con su chunk_id para poder citarlo.
    """
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(query, ticker=ticker, fiscal_year=fiscal_year,
                       item=item, k=k)
    )


# La primera llamada tarda unos segundos: carga el modelo de embeddings.
salida = search_filings.invoke({
    "query": "risks from misuse of our AI systems by third parties",
    "ticker": "MSFT", "fiscal_year": 2025, "item": "1A", "k": 3,
})
print(salida[:1200], "...")

# Dos cosas que mirar en esa salida:
#
# El corpus está EN INGLÉS. La consulta también tiene que ir en inglés aunque
# la pregunta del usuario venga en español. Que el agente traduzca la consulta
# es parte de su trabajo.
#
# Y el fragmento empieza a mitad de frase ("of operations."). Nadie ha decidido
# que empiece ahí: es donde cayó el corte del troceador. Eso es exactamente lo
# que se abre y se arregla el día 17.

[MSFT-2025-1A-0017] MSFT FY2025 Item 1A (similitud 0.842)
of operations.



Issues in the development, deployment, and use of AI may result in reputational or competitive harm or liability. We are building AI into many of our offerings, including our productivity services, and we are also making AI available for our customers to use in solutions that they build. This AI may be developed by Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to grow. We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate information. Content generated by AI systems may be offensive, illegal, inaccurate, or other

In [10]:
# La vía de contexto largo. Existe para que el agente pueda elegir pagarla.
@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA: puede devolver decenas de miles de tokens.
    Úsala solo cuando search_filings devuelva fragmentos insuficientes y
    necesites el contexto entero de una sección concreta.

    Args:
        ticker: Símbolo bursátil, p. ej. 'META'.
        fiscal_year: Ejercicio fiscal, p. ej. 2025.
        item: '1A' riesgos, '7' MD&A, '7A' riesgo de mercado,
            '8' estados financieros.
    """
    filas = secciones[(secciones.ticker == ticker)
                      & (secciones.fiscal_year == int(fiscal_year))
                      & (secciones.item == item)]
    if filas.empty:
        return (f"No hay Item {item} de {ticker} FY{fiscal_year} en el "
                f"corpus. Usa list_available para ver qué hay.")
    f = filas.iloc[0]
    return f.texto


# No la llamamos con el modelo delante: la sección más larga del corpus son
# 34.751 tokens y pagarlos para ver que funciona no tiene sentido. Miramos
# el tamaño de lo que devolvería.
for t, fy, it in [("AAPL", 2024, "7A"), ("META", 2025, "1A")]:
    texto = read_section.invoke(
        {"ticker": t, "fiscal_year": fy, "item": it})
    n = int(secciones[(secciones.ticker == t)
                      & (secciones.fiscal_year == fy)
                      & (secciones.item == it)].iloc[0].n_tokens)
    print(f"{t} FY{fy} Item {it}: {len(texto):,} caracteres, {n:,} tokens")

# Nada impide al agente llamar a read_section cuatro veces seguidas y
# quemarse el presupuesto del grupo en una pregunta. Hoy no hay nada que se
# lo impida; el día 17 se le pone un ToolCallLimitMiddleware.

AAPL FY2024 Item 7A: 3,043 caracteres, 612 tokens
META FY2025 Item 1A: 195,308 caracteres, 34,751 tokens


In [11]:
# EJERCICIO 1 (10 min) · list_available()  —— RESUELTO
#
# Sin esta herramienta el agente se inventa compañías y ejercicios que no
# están en el corpus: no tiene forma de comprobar el mundo, así que rellena
# el hueco con lo que le suena.
#
# Decisiones al escribirla, porque el enunciado del ejercicio deja margen:
#
# 1. **Devuelve texto y no un DataFrame.** Lo que devuelve una herramienta se
#    le pasa al modelo tal cual; un `repr` de pandas con columnas truncadas
#    es peor que una lista.
# 2. **Incluye la fecha de cierre de cada ejercicio.** No lo pide el
#    enunciado y arregla la trampa del §3: `fiscal_year` es el año en que
#    CIERRA el ejercicio, no el de presentación, y las seis compañías cierran
#    en cuatro meses distintos. Sin esta columna, el modelo tiene que
#    adivinarlo.
# 3. **Incluye los conceptos XBRL disponibles por compañía y ejercicio.**
#    Tampoco lo pide, y es la parte que más fallos evita: el concepto de los
#    ingresos NO es el mismo en todas las compañías, y el modelo, sin esta
#    lista, lo deduce por analogía con la anterior y se equivoca.
# 4. **El docstring dice CUÁNDO llamarla**, no qué hace. Es la parte del
#    ejercicio que de verdad cambia el comportamiento del agente.
@tool
def list_available() -> str:
    """Lista qué compañías, ejercicios fiscales y secciones existen en el corpus.

    ÚSALA SIEMPRE antes de afirmar que un dato no existe, y antes de cualquier
    otra herramienta si no estás seguro de que la compañía o el ejercicio por
    los que te preguntan están en el corpus. El corpus es cerrado y pequeño: si
    algo no aparece en esta lista, no está, y la respuesta correcta es decirlo
    con fuente="ninguna" en lugar de estimarlo.

    No necesita argumentos y es gratis: no cuesta ni una llamada al modelo ni
    una búsqueda.

    AVISO IMPORTANTE sobre el ejercicio fiscal: `fiscal_year` es el año en que
    CIERRA el ejercicio, no el año en que se presentó el informe. El FY2025 de
    NVIDIA cerró en enero de 2025 y el de Alphabet en diciembre de 2025.
    """
    lineas = ["COMPAÑÍAS Y EJERCICIOS DISPONIBLES", ""]
    for ticker, grupo in secciones.groupby("ticker"):
        cierres = (xbrl[xbrl.ticker == ticker]
                   .groupby("fiscal_year")["period_end"].max().to_dict())
        ejercicios = sorted(grupo.fiscal_year.unique())
        detalle = ", ".join(f"FY{fy} (cierra {cierres.get(fy, '?')})"
                            for fy in ejercicios)
        lineas.append(f"{ticker} — {grupo.iloc[0].empresa}")
        lineas.append(f"    ejercicios: {detalle}")
        lineas.append(f"    secciones : {sorted(grupo.item.unique())}")

    lineas += [
        "",
        "SECCIONES DEL 10-K",
        "    1A  Risk Factors — los riesgos que la compañía declara",
        "    7   MD&A — la dirección explicando sus propios resultados",
        "    7A  Market Risk — exposición a tipos, divisa y precios",
        "    8   Financial Statements — estados financieros y sus notas",
        "",
        "CONCEPTOS XBRL DISPONIBLES, POR COMPAÑÍA Y EJERCICIO",
        "(el concepto NO es el mismo en todas: compruébalo aquí antes de",
        " llamar a get_xbrl_fact, nunca lo deduzcas por analogía)",
        "",
    ]
    for (ticker, fy), grupo in xbrl.groupby(["ticker", "fiscal_year"]):
        lineas.append(f"    {ticker} FY{fy}: "
                      f"{', '.join(sorted(grupo.concept.unique()))}")
    return "\n".join(lineas)


print(list_available.invoke({}))

COMPAÑÍAS Y EJERCICIOS DISPONIBLES

AAPL — Apple Inc.
    ejercicios: FY2024 (cierra 2024-09-28), FY2025 (cierra 2025-09-27)
    secciones : ['1A', '7', '7A', '8']
AMZN — Amazon.com, Inc.
    ejercicios: FY2024 (cierra 2024-12-31), FY2025 (cierra 2025-12-31)
    secciones : ['1A', '7', '7A', '8']
GOOGL — Alphabet Inc.
    ejercicios: FY2024 (cierra 2024-12-31), FY2025 (cierra 2025-12-31)
    secciones : ['1A', '7', '7A', '8']
META — Meta Platforms, Inc.
    ejercicios: FY2024 (cierra 2024-12-31), FY2025 (cierra 2025-12-31)
    secciones : ['1A', '7', '7A', '8']
MSFT — Microsoft Corporation
    ejercicios: FY2024 (cierra 2024-06-30), FY2025 (cierra 2025-06-30)
    secciones : ['1A', '7', '7A', '8']
NVDA — NVIDIA Corporation
    ejercicios: FY2024 (cierra 2024-01-28), FY2025 (cierra 2025-01-26)
    secciones : ['1A', '7', '7A', '8']

SECCIONES DEL 10-K
    1A  Risk Factors — los riesgos que la compañía declara
    7   MD&A — la dirección explicando sus propios resultados
    7A  Market R

### Sobre la solución del ejercicio 1
>
> `list_available` es la única de las cuatro herramientas que no consulta nada:
> describe el mundo. Y es la que más fallos evita, porque casi todos los fallos
> caros de este dominio empiezan igual: el modelo no sabe que algo no existe, y
> rellena el hueco.
>
> La versión de arriba añade dos cosas que el enunciado del ejercicio no pedía,
> y conviene decir por qué:
>
> **La fecha de cierre de cada ejercicio.** El §3 del enunciado avisa de que
> `fiscal_year` es el año en que cierra el ejercicio y no el de presentación, y
> de que las seis compañías cierran en cuatro meses distintos. Esa información
> tiene que estar en algún sitio que el modelo pueda leer. Ponerla aquí cuesta
> tres líneas; no ponerla convierte la pregunta «¿el FY2025 de NVIDIA es el que
> cerró en enero de 2025?» en algo que el modelo tiene que adivinar.
>
> **Los conceptos XBRL disponibles, compañía por compañía.** Esta es la
> importante. El concepto de los ingresos no es universal: NVIDIA y Alphabet
> usan `Revenues`, y Apple, Microsoft, Meta y Amazon usan
> `RevenueFromContractWithCustomerExcludingAssessedTax`. Sin esta lista, el
> modelo que acaba de consultar Apple pide el mismo concepto para NVIDIA, no lo
> encuentra, y el fallo se manifiesta como una cifra inventada o como un bucle
> de reintentos. Con la lista, lo mira.
>
> El coste de esas dos adiciones es que la salida de la herramienta pasa de
> unas 400 palabras a unas 700, y se paga en tokens cada vez que el agente la
> llama. Compensa porque la llama como mucho una vez por pregunta y evita
> trayectorias enteras de reintentos, que son mucho más caras.

In [12]:
# El problema que resuelve: preguntar por algo que no está en el corpus.
#
# Tesla no está. La pregunta es qué hace el agente con eso.
PREGUNTA_FUERA = ("¿Cuál fue el revenue de Tesla en el ejercicio 2025 según "
                  "su 10-K?")

if modelo is not None:
    sin_lista = modelo.bind_tools([get_xbrl_fact, search_filings])
    con_lista = modelo.bind_tools([get_xbrl_fact, search_filings,
                                   list_available])

    for etiqueta, m in [("SIN list_available", sin_lista),
                        ("CON list_available", con_lista)]:
        r = m.invoke([{"role": "user", "content": PREGUNTA_FUERA}])
        print(f"\n--- {etiqueta} ---")
        if r.tool_calls:
            for tc in r.tool_calls:
                print(f"  llama a {tc['name']}({tc['args']})")
        else:
            print(f"  contesta directamente: {r.text[:200]}")
else:
    print("Sin clave: esta celda necesita el modelo. Seguid leyendo.")

# Lo que se ve casi siempre: sin la herramienta, el modelo llama a
# get_xbrl_fact con ticker='TSLA' y se queda esperando un dato que no existe,
# o directamente responde de memoria una cifra de Tesla que no ha salido de
# ningún informe. Con la herramienta, muchas veces comprueba primero.
#
# "Muchas veces" no es "siempre", y eso también es parte de la lección: una
# herramienta no es una garantía, es una opción que el modelo puede tomar.
# Convertirla en garantía es el trabajo del día 17 (guardrails).


--- SIN list_available ---
  llama a get_xbrl_fact({'ticker': 'TSLA', 'fiscal_year': 2025, 'concept': 'Revenues'})



--- CON list_available ---
  llama a list_available({})


In [13]:
# EJERCICIO 2 (10 min) · Dos docstrings, el mismo código  —— RESUELTO
#
# Abajo está get_xbrl_fact_vago: cuerpo idéntico al de get_xbrl_fact, con una
# descripción que no dice nada. Es la clase de docstring que se escribe cuando
# uno piensa que el docstring es documentación.
@tool
def get_xbrl_fact_vago(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve un dato financiero."""
    return get_xbrl_fact.func(ticker, fiscal_year, concept)


PREGUNTA = "¿Cuál fue el beneficio neto de Apple en el ejercicio 2025?"


def que_elige(herramientas, etiqueta: str) -> None:
    """Le hace la misma pregunta al modelo y enseña qué herramienta pide."""
    respuesta = modelo.bind_tools(herramientas).invoke(
        [{"role": "user", "content": PREGUNTA}])
    print(f"--- {etiqueta} ---")
    if not respuesta.tool_calls:
        print(f"  no llama a ninguna herramienta; contesta de memoria:")
        print(f"  {respuesta.text[:220]}")
        return
    for tc in respuesta.tool_calls:
        print(f"  llama a {tc['name']}({tc['args']})")


if modelo is not None:
    que_elige([get_xbrl_fact, search_filings, list_available],
              "docstring BUENO")
    print()
    que_elige([get_xbrl_fact_vago, search_filings, list_available],
              "docstring VAGO")
else:
    print("Sin clave: esta celda necesita el modelo.")

# Lo que hay que mirar no es solo QUÉ herramienta elige, sino con qué
# `concept` la llama.
#
# El docstring bueno enumera los conceptos de la taxonomía us-gaap, así que el
# modelo tiene de dónde sacar 'NetIncomeLoss'. El vago no dice ni que exista
# una taxonomía: el modelo tiene que adivinar el nombre exacto de un concepto
# contable a partir de la palabra "beneficio", y lo habitual es que invente
# algo plausible —'NetIncome', 'net_income', 'NetProfit'— que no existe.
#
# El cuerpo de las dos funciones es EL MISMO. La única diferencia es un texto
# que el modelo lee. Por eso el docstring de una herramienta no es
# documentación: es la única parte del programa donde reescribir un comentario
# cambia lo que hace el sistema.

# --- verificación de §3 --------------------------------------------------
assert "NVDA" in list_available.invoke({}), \
    "list_available debería nombrar las compañías del corpus."
assert "60,922,000,000" in get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"})
assert "no reportó" in get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}), \
    "La herramienta debe decir explícitamente que el concepto no está."
print("\n§3 listo: cuatro herramientas definidas.")

--- docstring BUENO ---
  llama a list_available({})



--- docstring VAGO ---
  llama a list_available({})

§3 listo: cuatro herramientas definidas.


## La descripción de una *tool* es *prompt engineering*, no documentación

El código de `get_xbrl_fact` y el de `get_xbrl_fact_vago` es exactamente el
mismo. Lo único que cambia es un párrafo de texto, y con él cambia el
comportamiento del agente. No hay ninguna otra parte del sistema donde
reescribir un comentario altere lo que hace el programa.

Tres consecuencias prácticas, que valen para el entregable:

1. **Decid cuándo llamarla, no solo qué hace.** «Es la fuente autorizada para
   cualquier cifra; úsala SIEMPRE en lugar de leer un número del texto» es
   una instrucción de enrutado.
2. **Decid cuándo *no* llamarla.** El docstring de `search_filings` dice que
   no se use para cifras. Esa frase vale más que las otras cinco.
3. **Poned el vocabulario dentro.** Si el parámetro espera `'1A'`, `'7'`,
   `'7A'` u `'8'`, el docstring tiene que enumerarlos: el modelo no puede
   adivinar un vocabulario que no ha visto.

---

## Pausa · 10 minutos

Al volver escribimos a mano el bucle que hace funcionar todo esto.

## `bind_tools`: qué devuelve el modelo cuando quiere una herramienta

Un modelo de lenguaje no ejecuta nada. Lo único que sabe hacer es producir
texto. Cuando decimos que «llama a una herramienta», lo que ocurre es esto:

1. Le mandamos los mensajes **y los esquemas** de las herramientas.
2. Él responde con una petición estructurada: nombre, argumentos y un `id`.
3. **Nosotros** ejecutamos la función.
4. Le devolvemos el resultado en un mensaje de tipo `tool`, con el mismo `id`.
5. Vuelta a empezar hasta que responda sin pedir nada.

Ese bucle de cinco pasos es todo. Lo importante es dónde está la frontera: el
paso 3 lo hace vuestro código, no el modelo. Un agente es un bucle `while`
alrededor de una llamada a un modelo, y quien ejecuta sois vosotros.

`model.bind_tools([...])` es lo que mete los esquemas en la petición.

In [14]:
# Qué devuelve exactamente el modelo cuando pide una herramienta.
HERRAMIENTAS = [list_available, get_xbrl_fact, search_filings, read_section]
POR_NOMBRE = {t.name: t for t in HERRAMIENTAS}

if modelo is not None:
    modelo_con_tools = modelo.bind_tools(HERRAMIENTAS)
    respuesta = modelo_con_tools.invoke([
        {"role": "user",
         "content": "¿Cuáles fueron los ingresos de NVIDIA en FY2025?"},
    ])

    print("texto de la respuesta:", repr(respuesta.text))
    print("\ntool_calls, en crudo:")
    for tc in respuesta.tool_calls:
        print(f"  name : {tc['name']}")
        print(f"  args : {tc['args']}")
        print(f"  id   : {tc['id']}")
else:
    modelo_con_tools = None
    print("Sin clave: esta celda necesita el modelo.")

# El texto viene vacío o casi. El contenido de la respuesta ES la petición de
# herramienta. Y el `id` no es decorativo: es lo que empareja cada resultado
# con su petición cuando hay varias a la vez.

texto de la respuesta: ''

tool_calls, en crudo:
  name : list_available
  args : {}
  id   : call_TtAqQxnYK2wBNdPToWc1ehZV


In [15]:
# EL BUCLE. Treinta líneas, sin framework.  —— RESUELTO
from langchain.messages import ToolMessage

SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
"""


def agente_manual(pregunta: str, max_vueltas: int = 6,
                  verboso: bool = True) -> str:
    mensajes = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pregunta}]
    for vuelta in range(max_vueltas):
        respuesta = modelo_con_tools.invoke(mensajes)
        mensajes.append(respuesta)

        if not respuesta.tool_calls:
            return respuesta.text

        for tc in respuesta.tool_calls:
            if verboso:
                argumentos = ", ".join(f"{k}={v!r}"
                                       for k, v in tc["args"].items())
                print(f"  vuelta {vuelta + 1}: {tc['name']}({argumentos})")
            try:
                resultado = POR_NOMBRE[tc["name"]].invoke(tc["args"])
            except KeyError:
                # El modelo puede pedir una herramienta que no existe. Es un
                # error suyo, y la forma de que lo corrija es decírselo, no
                # reventar: el mensaje vuelve al modelo como información.
                resultado = (
                    f"ERROR: no existe ninguna herramienta llamada "
                    f"'{tc['name']}'. Las disponibles son: "
                    f"{', '.join(POR_NOMBRE)}."
                )
            except Exception as e:
                # Lo mismo con cualquier otro fallo: argumentos mal tipados,
                # un filtro imposible, un fichero que no está. Un agente que
                # revienta con la primera excepción no llega a la sesión 2.
                resultado = (
                    f"ERROR al ejecutar {tc['name']}: {type(e).__name__}: {e}. "
                    f"Revisa los argumentos y vuelve a intentarlo, o prueba "
                    f"con otra herramienta."
                )

            # El tool_call_id tiene que ser el de ESTA petición: es lo que
            # empareja cada resultado con su pregunta cuando el modelo pide
            # varias herramientas en la misma vuelta. Si se cruzan, el modelo
            # lee la respuesta de otra pregunta y no da ningún error.
            mensajes.append(ToolMessage(
                content=str(resultado),
                tool_call_id=tc["id"],
                name=tc["name"],
            ))
            if verboso:
                recorte = str(resultado).replace("\n", " ")[:150]
                print(f"       -> {recorte}…")

    return "Se agotaron las vueltas sin llegar a una respuesta."


print("agente_manual definido.")

# Tres detalles del cuerpo que no son opcionales:
#
# 1. **El resultado vuelve como ToolMessage, no como texto del usuario.** El
#    modelo distingue los dos: un ToolMessage es «esto lo ha producido la
#    herramienta que pediste», y es lo que le permite encadenar.
# 2. **Las excepciones se le devuelven al modelo.** Es contraintuitivo la
#    primera vez: parece que un error debería propagarse. Pero el modelo puede
#    corregirse si sabe qué pasó, y no puede hacer nada si el proceso ha
#    muerto.
# 3. **Se recorren TODAS las tool_calls de la vuelta.** El modelo puede pedir
#    dos herramientas a la vez —típico en una comparativa: los dos ejercicios
#    en paralelo— y dejarse una sin contestar deja la conversación en un
#    estado que la API rechaza.

agente_manual definido.


### Sobre la solución del bucle: por qué los errores vuelven al modelo
>
> La parte del bucle que más se resiste la primera vez no es el `ToolMessage`:
> es qué hacer cuando la herramienta lanza una excepción.
>
> El instinto de cualquiera que programe es dejar que el error se propague. En
> un agente eso es casi siempre lo incorrecto, y la razón es que **el modelo es
> capaz de corregirse si se entera de lo que ha pasado, y no puede hacer
> absolutamente nada si el proceso ha muerto.** Un `concept` mal escrito, un
> filtro que no devuelve nada, un `fiscal_year` como cadena en vez de entero:
> los tres son errores recuperables, y la forma de recuperarlos es devolverle
> al modelo un texto que diga qué falló y qué puede probar.
>
> Lo mismo, en otra escala, es lo que hacen los docstrings de las herramientas
> cuando no hay resultados: un mensaje que dice qué cambiar, en lugar de una
> cadena vacía que invita a repetir la misma llamada.
>
> Hay un límite, y está en la celda siguiente: un agente que siempre puede
> reintentar es un agente que puede no parar nunca. La red de seguridad no es
> el manejo de errores, es el tope de vueltas, y el día 17 se formaliza con
> `ToolCallLimitMiddleware`.

In [16]:
# Una pregunta que no se contesta con una sola herramienta: hace falta el
# texto (qué riesgo nuevo) y la cifra (cuánto creció). Dos herramientas
# distintas, y el modelo tiene que decidir el orden.
PREGUNTA_DOBLE = (
    "¿Qué riesgos nuevos relacionados con la inteligencia artificial añadió "
    "Microsoft en su 10-K de FY2025 respecto al de FY2024, y cuánto creció su "
    "revenue entre esos dos ejercicios?"
)

if modelo_con_tools is not None:
    print("TRAYECTORIA")
    final = agente_manual(PREGUNTA_DOBLE, max_vueltas=8)
    print("\nRESPUESTA\n", final)
else:
    print("Sin clave: esta celda necesita el modelo.")

# Comparad la trayectoria con la de la demo del principio de la sesión.
# Es la misma, y acabáis de escribirla.

# --- verificación de §5 --------------------------------------------------
import inspect
assert "ToolMessage" in inspect.getsource(agente_manual), \
    "El bucle tiene que devolverle el resultado al modelo con un ToolMessage."
assert "tool_call_id" in inspect.getsource(agente_manual), \
    "Cada ToolMessage necesita el tool_call_id de su petición."
print("\n§5 listo.")

TRAYECTORIA


  vuelta 1: list_available()
       -> COMPAÑÍAS Y EJERCICIOS DISPONIBLES  AAPL — Apple Inc.     ejercicios: FY2024 (cierra 2024-09-28), FY2025 (cierra 2025-09-27)     secciones : ['1A', '7…


  vuelta 2: search_filings(query='artificial intelligence OR AI', ticker='MSFT', fiscal_year=2024, item='1A', k=10)
       -> [MSFT-2024-1A-0017] MSFT FY2024 Item 1A (similitud 0.695) .  Issues in the development and use of AI may result in reputational or competitive harm or…
  vuelta 2: search_filings(query='artificial intelligence OR AI', ticker='MSFT', fiscal_year=2025, item='1A', k=10)
       -> [MSFT-2025-1A-0017] MSFT FY2025 Item 1A (similitud 0.700) of operations.    Issues in the development, deployment, and use of AI may result in reputat…
  vuelta 2: get_xbrl_fact(ticker='MSFT', fiscal_year=2024, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 245,122,000,000 USD (cierre de ejercicio 2024-06-30, según el 10-K)…
  vuelta 2: get_xbrl_fact(ticker='MSFT', fiscal_year=2025, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2025 · RevenueFromContractWithCustom

  vuelta 3: search_filings(query='agentic', ticker='MSFT', fiscal_year=2024, item='1A', k=10)
       -> [MSFT-2024-1A-0015] MSFT FY2024 Item 1A (similitud 0.534) Advertising, professional, marketplace, and gaming platform abuses  For platform products an…


  vuelta 4: get_xbrl_fact(ticker='MSFT', fiscal_year=2024, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 245,122,000,000 USD (cierre de ejercicio 2024-06-30, según el 10-K)…
  vuelta 4: get_xbrl_fact(ticker='MSFT', fiscal_year=2025, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> MSFT FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 281,724,000,000 USD (cierre de ejercicio 2025-06-30, según el 10-K)…



RESPUESTA
 Resumen breve

- Nuevos riesgos/énfasis añadidos en FY2025 respecto a FY2024:
  1. Mención explícita de “agentic AI” — sistemas de IA que pueden actuar de forma autónoma y la necesidad de revisión humana de entradas y salidas en esos escenarios (nuevo lenguaje en FY2025). Fuente: MSFT FY2025 Item 1A, chunk_id = MSFT-2025-1A-0017.  
  2. Regulación enfocada en “frontier model safety”, transparencia y la procedencia del contenido (content provenance) como áreas regulatorias emergentes — riesgo regulatorio ampliado en FY2025. Fuente: MSFT FY2025 Item 1A, chunk_id = MSFT-2025-1A-0022.  
  3. Riesgos relacionados con controles de exportación de IA y medidas como la “AI Diffusion Rule” y su impacto en el comercio y la cadena de suministro (mayor énfasis geopolítico y de comercio/controles de exportación en FY2025). Fuente: MSFT FY2025 Item 1A, chunk_id = MSFT-2025-1A-0018.

- Contexto: FY2024 ya describía riesgos generales asociados al desarrollo y uso de IA (algoritmos defectuos

In [17]:
# EL BUCLE INFINITO. Esta celda está diseñada para portarse mal.
#
# La pregunta pide el margen bruto de Amazon. Amazon NO etiqueta GrossProfit
# en us-gaap, así que get_xbrl_fact devuelve un aviso una y otra vez, y el
# modelo tiende a reintentar con variantes del concepto en lugar de rendirse.
#
# Con max_vueltas=100 no hay nada que lo pare. Dejadlo correr unas cuantas
# vueltas para verlo y CORTAD LA CELDA A MANO (el cuadrado de stop).
#
# Ojo: cada vuelta cuesta dinero del presupuesto del grupo. Diez vueltas
# bastan para entenderlo.
PREGUNTA_BUCLE = ("Compara el margen bruto de Amazon en FY2024 y FY2025 y "
                  "explica a qué se debe el cambio.")

if modelo_con_tools is not None:
    print(agente_manual(PREGUNTA_BUCLE, max_vueltas=100))
else:
    print("Sin clave: esta celda necesita el modelo.")

# NO lo arreglamos hoy. El día 17 se cierra con una línea:
#     ToolCallLimitMiddleware(run_limit=8)
# El valor de este bloque está en haber visto el problema una semana antes de
# ver la solución.

  vuelta 1: list_available()
       -> COMPAÑÍAS Y EJERCICIOS DISPONIBLES  AAPL — Apple Inc.     ejercicios: FY2024 (cierra 2024-09-28), FY2025 (cierra 2025-09-27)     secciones : ['1A', '7…


  vuelta 2: get_xbrl_fact(ticker='AMZN', fiscal_year=2024, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> AMZN FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 637,959,000,000 USD (cierre de ejercicio 2024-12-31, según el 10-K)…


  vuelta 3: get_xbrl_fact(ticker='AMZN', fiscal_year=2024, concept='CostOfRevenue')
       -> AMZN no reportó 'CostOfRevenue' en FY2024. Conceptos disponibles: Assets, EarningsPerShareDiluted, IncomeTaxExpenseBenefit, NetCashProvidedByUsedInOpe…


  vuelta 4: get_xbrl_fact(ticker='AMZN', fiscal_year=2025, concept='RevenueFromContractWithCustomerExcludingAssessedTax')
       -> AMZN FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 716,924,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)…


  vuelta 5: search_filings(query='gross margin', ticker='AMZN', fiscal_year=2025, item='7', k=5)
       -> [AMZN-2025-7-0001] AMZN FY2025 Item 7 (similitud 0.651) Overview  Our primary source of revenue is the sale of a wide range of products and services t…


  vuelta 6: search_filings(query='gross profit', ticker='AMZN', fiscal_year=2025, item='7', k=10)
       -> [AMZN-2025-7-0001] AMZN FY2025 Item 7 (similitud 0.698) Overview  Our primary source of revenue is the sale of a wide range of products and services t…


Resumen clave
- No puedo calcular ni comparar numéricamente el margen bruto de Amazon porque en el XBRL del 10‑K de Amazon no se reportan ni GrossProfit ni CostOfRevenue para FY2024/FY2025. (XBRL: la consulta a CostOfRevenue devolvió “no reportó”.) Fuente de esta ausencia: datos XBRL del corpus (no disponible).  
- Puedo, en cambio, confirmar las ventas reportadas por XBRL: FY2024 = 637,959,000,000 USD; FY2025 = 716,924,000,000 USD (ambas cifras obtenidas mediante get_xbrl_fact).

Por qué no puedo dar el cambio de margen bruto
- Regla del análisis: para cualquier cifra debo usar get_xbrl_fact. Como los conceptos GrossProfit o CostOfRevenue no existen en el XBRL disponible para AMZN en esos ejercicios, no hay cifra autorizada en el corpus para calcular margen bruto (no puedo usar ni extrapolar números que aparecen solo en la narrativa). Fuente: resultado de la consulta XBRL a CostOfRevenue (no reportado).

Explicación cualitativa — qué dice la compañía sobre las causas del cambio
- Aume

## Lo que acabáis de escribir tiene nombre

Se llama **ReAct** (*Reasoning + Acting*), es de 2022, y es el patrón sobre el
que está construida la mayor parte de los agentes que hay hoy en producción.
El bucle de la celda anterior es, sin quitar ni añadir nada: razonar, actuar,
observar, repetir.

**El viernes 18 os lo van a explicar con el paper delante.** Hoy lo habéis
escrito. Cuando lleguéis a esa clase, no vais a estar aprendiendo un patrón
nuevo: vais a estar poniéndole nombre a las treinta líneas que ya tenéis.

Ese es también el motivo de que el bloque siguiente vaya después y no antes.
`create_agent` hace esto mismo en cinco líneas, y solo se aprecia lo que
resuelve si antes lo habéis escrito a mano.

In [18]:
# El esquema de respuesta. Es el contrato §7 del enunciado, literal.
#
# Fijaos en lo que hace: convierte la cita de una súplica en el prompt
# ("por favor, cita la fuente") en un requisito estructural. El modelo no
# puede devolver una respuesta sin decir de dónde sale, porque el esquema no
# valida. Y los evaluadores del día 17 leen campos en lugar de parsear prosa.
from typing import Literal

from pydantic import BaseModel, Field


class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""

    respuesta: str = Field(
        description="Respuesta en prosa, breve y directa")
    cifra: float | None = Field(
        default=None, description="Valor numérico, si la pregunta pide uno")
    unidad: str | None = Field(
        default=None, description="USD, shares, porcentaje…")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description="De dónde sale el dato. 'ninguna' si no está en el corpus")
    cita: str | None = Field(
        default=None,
        description="Texto literal del informe que respalda la respuesta")
    chunk_id: str | None = Field(
        default=None,
        description="Identificador del fragmento citado, para verificar")


print(json.dumps(RespuestaFinanciera.model_json_schema()["properties"],
                 indent=2, ensure_ascii=False)[:600], "...")

{
  "respuesta": {
    "description": "Respuesta en prosa, breve y directa",
    "title": "Respuesta",
    "type": "string"
  },
  "cifra": {
    "anyOf": [
      {
        "type": "number"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Valor numérico, si la pregunta pide uno",
    "title": "Cifra"
  },
  "unidad": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "USD, shares, porcentaje…",
    "title": "Unidad"
  },
  "ticker": {
    "anyOf": [
      ...


### Nota sobre el esquema de respuesta que se entrega
>
> La `RespuestaFinanciera` de arriba es la del §7 del enunciado, literal, y así
> se queda en este notebook.
>
> En el paquete `agente/esquema.py` hay una versión con tres campos más
> —`cifra_anterior`, `ejercicio_anterior` y `concepto_xbrl`—, todos opcionales
> y con valor por defecto, de modo que sigue cumpliendo el contrato original.
> Los tres existen por el mismo motivo: **hacen verificable algo que de otro
> modo habría que creerse.**
>
> - `cifra_anterior` y `ejercicio_anterior` permiten distinguir a un agente que
>   consultó los dos ejercicios de uno que consultó uno y estimó el otro. Sin
>   ellos, una comparativa con la cifra reciente correcta parece bien resuelta
>   aunque la variación esté inventada.
> - `concepto_xbrl` deja por escrito qué concepto se consultó, y con eso el
>   guardrail del día 17 puede comparar contra **ese** concepto en lugar de
>   contra el conjunto de todos los hechos de la compañía. Es la diferencia
>   entre detectar «esta cifra no es de ninguna magnitud reportada» y detectar
>   «esta cifra es el ingreso, y te preguntaban por el beneficio».

In [19]:
# El mismo agente, en cinco líneas.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=config.crear_modelo(),
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
    )
    print("Agente montado con", len(HERRAMIENTAS), "herramientas.")
else:
    print("Sin clave: no se puede montar el agente.")

# Qué hace cada línea que vosotros hicisteis a mano:
#
#   tools=            los bind_tools y el diccionario POR_NOMBRE
#   response_format=  la validación de la salida, que no teníais
#   checkpointer=     la memoria entre invocaciones, que tampoco teníais
#   (el bucle)        las treinta líneas de la celda anterior
#
# Y trae cosas que no habíais escrito: reintentos, streaming, callbacks para
# instrumentar la ejecución, e interrupciones para humano en el medio, que es
# de lo que va el día 17.
#
# Se le pasa `config.crear_modelo()` y no la cadena `"openai:..."` por una
# razón concreta: `crear_modelo()` fija `temperature=0` solo en las familias
# que lo admiten. La familia gpt-5 devuelve un 400 si se le pasa, y pasar la
# cadena a pelo no deja sitio donde poner esa excepción.
#
# Si el modelo que elijáis no soporta salida estructurada nativa, esa línea
# falla. La salida es envolverlo:
#     from langchain.agents.structured_output import ToolStrategy
#     response_format=ToolStrategy(schema=RespuestaFinanciera)

Agente montado con 4 herramientas.


In [20]:
# Memoria: dos invocaciones con el mismo thread_id.
#
# `thread_id` es lo que convierte dos llamadas sueltas en una conversación.
# Sin checkpointer no hay memoria; sin thread_id, el checkpointer no sabe
# a qué conversación pertenece cada invocación.
CONFIG = {"configurable": {"thread_id": "clase-s1"}}

if agente is not None:
    r1 = agente.invoke(
        {"messages": [{"role": "user", "content":
                       "¿Cuál fue el revenue de NVIDIA en FY2025?"}]},
        config=CONFIG,
    )
    print("1 >", r1["structured_response"].respuesta)

    # Esta segunda pregunta no nombra ni la compañía ni el ejercicio.
    r2 = agente.invoke(
        {"messages": [{"role": "user",
                       "content": "¿Y cuánto es eso comparado con el "
                                  "ejercicio anterior?"}]},
        config=CONFIG,
    )
    print("2 >", r2["structured_response"].respuesta)
    print(f"\nmensajes acumulados en el hilo: {len(r2['messages'])}")
else:
    r1 = r2 = None
    print("Sin clave: esta celda necesita el agente.")

# Cambiad el thread_id de la segunda invocación y volved a ejecutar: la
# pregunta de seguimiento deja de tener sentido. Eso es exactamente lo que
# le pasa a un agente sin memoria.

Deserializing unregistered type __main__.RespuestaFinanciera from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'RespuestaFinanciera')]


1 > El revenue (Revenues) reportado por NVIDIA en FY2025 fue 130,497,000,000 USD.


2 > En FY2025 NVIDIA reportó 130,497,000,000 USD frente a 60,922,000,000 USD en FY2024: un aumento absoluto de 69,575,000,000 USD, que equivale a un incremento de ≈114.18%.

mensajes acumulados en el hilo: 10


In [21]:
# Ver la trayectoria.
#
# Una respuesta no se puede juzgar sin ver el camino. Esta función imprime la
# trayectoria: qué herramientas se llamaron, en qué orden, con qué argumentos
# y qué devolvió cada una.
#
# Es también la pieza que hace posible el evaluador `uso_la_tool_correcta` del
# día 17: sin trayectoria no se puede distinguir una respuesta correcta de una
# respuesta correcta por casualidad.
def pretty_trace(resultado) -> None:
    """Imprime la trayectoria: qué herramientas se llamaron, con qué
    argumentos y qué devolvieron."""
    pendientes = {}
    n = 0
    for mensaje in resultado["messages"]:
        for tc in getattr(mensaje, "tool_calls", None) or []:
            n += 1
            pendientes[tc["id"]] = n
            print(f"{n}. {tc['name']}({tc['args']})")
        id_llamada = getattr(mensaje, "tool_call_id", None)
        if id_llamada in pendientes:
            texto = str(mensaje.content).replace("\n", " ")
            print(f"   -> {texto[:160]}"
                  f"{'...' if len(texto) > 160 else ''}")
    print(f"\n{n} llamadas a herramienta")
    e = resultado.get("structured_response")
    if e is not None:
        print(f"respuesta: {e.respuesta}")
        print(f"fuente: {e.fuente} · cifra: {e.cifra} {e.unidad or ''} "
              f"· cita: {e.chunk_id}")


if r2 is not None:
    pretty_trace(r2)

1. list_available({})
   -> COMPAÑÍAS Y EJERCICIOS DISPONIBLES  AAPL — Apple Inc.     ejercicios: FY2024 (cierra 2024-09-28), FY2025 (cierra 2025-09-27)     secciones : ['1A', '7', '7A', '...
2. get_xbrl_fact({'ticker': 'NVDA', 'fiscal_year': 2025, 'concept': 'Revenues'})
   -> NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)
3. get_xbrl_fact({'ticker': 'NVDA', 'fiscal_year': 2024, 'concept': 'Revenues'})
   -> NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)

3 llamadas a herramienta
respuesta: En FY2025 NVIDIA reportó 130,497,000,000 USD frente a 60,922,000,000 USD en FY2024: un aumento absoluto de 69,575,000,000 USD, que equivale a un incremento de ≈114.18%.
fuente: xbrl · cifra: 114.18 % · cita: None


In [22]:
# --- verificación de §6 --------------------------------------------------
if agente is None:
    print("Sin clave: §6 no se puede verificar. El resto del notebook sí.")
else:
    prueba = agente.invoke(
        {"messages": [{"role": "user", "content":
                       "¿Cuál fue el revenue de NVIDIA en FY2024?"}]},
        config={"configurable": {"thread_id": "verificacion-s1"}},
    )
    e = prueba["structured_response"]

    assert isinstance(e, RespuestaFinanciera), \
        "structured_response debería ser una RespuestaFinanciera validada."
    assert e.fuente in {"xbrl", "texto", "ambas", "ninguna"}
    assert any(
        tc["name"] == "get_xbrl_fact"
        for m in prueba["messages"]
        for tc in (getattr(m, "tool_calls", None) or [])
    ), ("Una pregunta numérica que no pasó por get_xbrl_fact es un fallo, "
        "aunque el número sea correcto. Revisad el system prompt.")

    print("§6 listo:", e.respuesta)
    print(f"cifra={e.cifra} unidad={e.unidad} fuente={e.fuente}")

§6 listo: El revenue (Revenues) reportado por NVIDIA en FY2024 fue 60,922,000,000 USD.
cifra=60922000000.0 unidad=USD fuente=xbrl


## La práctica

El enunciado completo lo tenéis en la mano. Aquí queda lo imprescindible.

**Qué se entrega**, en grupos de 3:

| | Peso |
| --- | --- |
| Repositorio con el agente, el golden set y los evaluadores | 30 |
| Presentación de 8 minutos el 24, con 10 preguntas ciegas en directo | 70 |

**El golden set.** Recibís 20 preguntas oficiales y escribís **20 vuestras**,
de las cuales **al menos 6 comparativas**. Tres familias:

| Familia | Qué mide | Campos que se rellenan |
| --- | --- | --- |
| `extractiva` | Retrieval y trazabilidad | `item_esperado`, `ancla_texto` |
| `numerica` | El guardrail contra XBRL | `cifra_esperada`, `unidad`, `concept_xbrl` |
| `comparativa` | Que el agente descomponga y compare | los tres, por cada ejercicio |

**Por qué el mínimo de 6 comparativas.** «¿Qué riesgos nuevos añadió entre
FY2024 y FY2025?» no la contesta una sola recuperación: hay que descomponer,
recuperar dos veces y comparar. Es la familia donde el agente deja de ser
decoración sobre una *pipeline*, y sin ella vuestro informe no puede demostrar
que hiciera falta un agente. Hay sustancia real que encontrar: el Item 1A
cambia entre el 49 % y el 85 % de los párrafos entre los dos ejercicios, según
la compañía.

**La verdad se ancla a una frase, no a un `chunk_id`.** El día 17 vais a
cambiar el troceado, y en cuanto lo toquéis todos los `chunk_id` son otros. Si
la métrica dependiera de ellos, el grupo que mejorase el troceado saldría
penalizado por haberlo mejorado. Por eso `ancla_texto` es un texto literal del
informe: **una frase**, no tres párrafos.

In [23]:
# El golden set oficial: las 20 preguntas con respuesta conocida.
from pathlib import Path

RUTA_GOLDEN = config.RUTA_GOLDEN_OFICIAL
if not RUTA_GOLDEN.is_file():
    RUTA_GOLDEN = config.RUTA_GOLDEN_EJEMPLO   # provisional

golden = [json.loads(l) for l in open(RUTA_GOLDEN, encoding="utf-8")
          if l.strip()]
g = pd.DataFrame(golden)

print(f"{RUTA_GOLDEN.name}: {len(g)} preguntas")
print("\nreparto por familia:")
print(g.familia.value_counts().to_string())
print("\nherramienta esperada:")
print(g.herramienta_esperada.explode().value_counts().to_string())

for fila in golden[:2]:
    print("\n" + json.dumps(fila, ensure_ascii=False, indent=2))

# NOTA PARA EL 10 DE SEPTIEMBRE: si arriba pone `golden_set_ejemplo.jsonl`,
# es que las 20 oficiales todavía no están en la carpeta. El fichero de
# ejemplo tiene tres preguntas y sirve solo para ver el esquema y probar el
# validador de la celda siguiente.

golden_set.jsonl: 20 preguntas

reparto por familia:
familia
numerica       7
comparativa    7
extractiva     6

herramienta esperada:
herramienta_esperada
get_xbrl_fact     14
search_filings    13

{
  "id": "of-001",
  "pregunta": "¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?",
  "familia": "extractiva",
  "ticker": "NVDA",
  "fiscal_year": 2025,
  "respuesta_esperada": "Que el mercado chino, donde sus productos están limitados por los controles de exportación, es muy competitivo y espera que lo siga siendo.",
  "cifra_esperada": null,
  "unidad": null,
  "concept_xbrl": null,
  "item_esperado": "1A",
  "ancla_texto": "The market in China, where our offerings are limited by export controls, is highly competitive and we expect it to remain competitive going forward.",
  "ancla_inicio": 31820,
  "ancla_fin": 31968,
  "chunk_id_esperado": "NVDA-2025-1A-0013",
  "herramienta_esperada": [
    "search_filings"
  ],
  "autor"

In [24]:
# Vuestras 20 preguntas, y el validador que tienen que pasar antes de
# entregarlas. Un golden set que no pasa el validador no se corrige: se
# devuelve.
PLANTILLA = {
    "id": "g3-001",
    "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
    "familia": "numerica",              # extractiva | numerica | comparativa
    "ticker": "NVDA",
    "fiscal_year": 2024,
    "respuesta_esperada": "60.922 millones de dólares",
    "cifra_esperada": 60922000000.0,
    "unidad": "USD",
    "concept_xbrl": "Revenues",
    "item_esperado": None,
    "ancla_texto": None,
    "ancla_inicio": None,
    "ancla_fin": None,
    "chunk_id_esperado": None,
    "herramienta_esperada": ["get_xbrl_fact"],
    "autor": "grupo-3",
}

CAMPOS = set(PLANTILLA)
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    """Los problemas del fichero, uno por línea. Lista vacía = correcto."""
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el "
                             f"corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(
                    f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                    f"FY{p['fiscal_year']}. El concepto se mira en "
                    f"xbrl_facts.parquet, nunca por analogía con otra "
                    f"compañía.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(
                    f"{pid}: ancla de {len(ancla.split())} palabras. Una "
                    f"frase. Así no medís vuestro retrieval, medís vuestro "
                    f"tamaño de ventana.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas


problemas = validar(golden, exigir_20=False)
print("Validando el fichero cargado:")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")

# --- verificación de §7 --------------------------------------------------
assert not validar([PLANTILLA], exigir_20=False), \
    "La plantilla debería pasar su propio validador."
assert validar([{**PLANTILLA, "ticker": "TSLA"}], exigir_20=False), \
    "El validador tiene que rechazar una compañía que no está en el corpus."
print("\n§7 listo. El validador funciona; ahora escribid las preguntas.")

Validando el fichero cargado:
  sin problemas

§7 listo. El validador funciona; ahora escribid las preguntas.


### Cierre del §7: las 20 preguntas propias pasan **este** validador
>
> El validador de la celda anterior es el de clase, escrito aquí, con sus
> variables locales (`secciones`, `xbrl`) y sus reglas. En el paquete hay una
> copia ampliada, `agente.esquema.validar_golden`, que el notebook
> `03_Golden_set_propio.ipynb` usa para construir el fichero.
>
> Que la copia ampliada dé el visto bueno no demuestra nada si la original no
> lo da: sería como corregirse el examen uno mismo. La celda de abajo pasa
> `golden_set_propio.jsonl` por el validador **de esta celda**, sin tocarlo, y
> exige cero problemas. Es la comprobación que cierra el §7 del enunciado.
>
> Las preguntas de ausencia van en un fichero aparte, `golden_set_ausencias.jsonl`,
> y **no** se pasan por aquí, a propósito: este validador exige
> `cifra_esperada` en la familia numérica, y una pregunta cuya respuesta
> correcta es «ese dato no está en el corpus» no tiene cifra que esperar.
> Forzarlas a cumplirlo obligaría a inventarse un número, que es justo lo
> contrario de lo que esas preguntas miden. El razonamiento completo está en
> el notebook 03.

In [25]:
# El golden set propio, contra el validador de la celda anterior.
RUTA_PROPIO = config.RUTA_GOLDEN_PROPIO

if not RUTA_PROPIO.is_file():
    print(f"Todavía no existe {RUTA_PROPIO}. "
          f"Ejecuta 03_Golden_set_propio.ipynb.")
else:
    propio = [json.loads(l) for l in open(RUTA_PROPIO, encoding="utf-8")
              if l.strip()]
    gp = pd.DataFrame(propio)

    print(f"{RUTA_PROPIO.name}: {len(propio)} preguntas, "
          f"autor '{gp.autor.iloc[0]}'")
    print("\nreparto por familia:")
    print(gp.familia.value_counts().to_string())
    print("\ncobertura: "
          f"{gp.ticker.nunique()}/6 compañías, "
          f"{gp.fiscal_year.nunique()}/2 ejercicios, "
          f"{gp.item_esperado.dropna().nunique()}/4 items")
    print("\nherramienta esperada:")
    print(gp.herramienta_esperada.explode().value_counts().to_string())

    problemas_propio = validar(propio, exigir_20=True)
    print("\nValidador de clase:")
    print("\n".join(f"  - {p}" for p in problemas_propio) or "  sin problemas")
    assert not problemas_propio, \
        "El golden set propio tiene que pasar el validador de clase tal cual."

    # Y las de ausencia, contadas pero no validadas aquí. Ver la celda de
    # arriba para el motivo.
    ausencias = [json.loads(l) for l in
                 open(config.RUTA_GOLDEN_AUSENCIAS, encoding="utf-8")
                 if l.strip()]
    print(f"\nMás {len(ausencias)} preguntas de ausencia en "
          f"golden_set_ausencias.jsonl, evaluadas aparte con el criterio "
          f"fuente == 'ninguna' y cifra is None.")

    print("\n§7 cerrado: 20 preguntas propias válidas y "
          f"{int((gp.familia == 'comparativa').sum())} comparativas "
          f"(el mínimo son 6).")

golden_set_propio.jsonl: 20 preguntas, autor 'grupo-nlp'

reparto por familia:
familia
extractiva     7
numerica       7
comparativa    6

cobertura: 6/6 compañías, 2/2 ejercicios, 4/4 items

herramienta esperada:
herramienta_esperada
search_filings    13
get_xbrl_fact     13

Validador de clase:
  sin problemas

Más 7 preguntas de ausencia en golden_set_ausencias.jsonl, evaluadas aparte con el criterio fuente == 'ninguna' y cifra is None.

§7 cerrado: 20 preguntas propias válidas y 6 comparativas (el mínimo son 6).


## Para el jueves 17

Dos cosas, y sin ellas la sesión que viene no os rinde:

1. **El baseline corriendo.** Este notebook, ejecutado de arriba abajo en
   vuestro repositorio, con `responder()` y `evaluar()` como dice el §6 del
   enunciado. El día 17 empieza ejecutando vuestro agente contra cinco
   preguntas duras y viendo en qué falla: si llegáis sin agente, la primera
   hora se convierte en soporte técnico y la perdéis.
2. **Vuestras 20 preguntas escritas**, pasando el validador de la celda
   anterior, con al menos 6 comparativas.

> Si el baseline no os arranca el día 17, hay un notebook de rescate que lo
> reconstruye en tres minutos. Existe para que nadie se quede fuera, no para
> ahorrarse el trabajo: quien lo use empieza la sesión sin conocer su propio
> código.

## Qué se lleva de hoy

- Una herramienta no es una función: es una **función más un docstring**, y el
  docstring es lo que decide si se llama. La descripción es *prompt
  engineering*, no documentación.
- Un agente es un **bucle `while`** alrededor de una llamada a un modelo. Lo
  habéis escrito. `create_agent` hace lo mismo con reintentos, memoria,
  validación y trazas.
- El retrieval no es la arquitectura: es **una herramienta más**, y compite
  con una consulta a XBRL que es mil veces más barata y exacta. Enrutar bien
  entre las dos es el trabajo.
- Un agente sin límites entra en bucle. Lo habéis visto. El día 17 se arregla.

## Qué falta, y cuándo llega

| Falta | Cuándo |
| --- | --- |
| Qué hay dentro de `search_filings`, y cómo medirlo | 17 sep |
| Que el agente no se invente cifras | 17 sep |
| Cortar el bucle infinito | 17 sep |
| Evaluar de verdad, con el golden set | 17 sep |
| ReAct, el paper y el nombre de lo que habéis escrito | 18 sep |
| MCP, ADK y A2A | 19 sep, más un notebook de bonus |